# NBA Live Win Probability Model (XGBoost)

Goal: given the in-game state at any moment (score, time remaining, possession, matchup strength, etc.), predict the probability the **home team** wins.

Pipeline:
1. Load data and inspect features
2. Split by `game_id` (not by row) so no game leaks across train/test
3. Hyperparameter search with `GroupKFold` cross-validation
4. Fit final XGBoost model on the best params
5. Evaluate (log loss, AUC, Brier score, calibration curve)
6. Wrap everything into a single `predict_win_probability(...)` function
7. Sanity check by plotting the predicted win probability trajectory for one game


In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold, RandomizedSearchCV, GroupShuffleSplit
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve

import joblib

pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42


## 1. Load data

Assumes you've already read the CSV into `df` (as in your prompt). If you're running
this notebook standalone, uncomment the line below.


In [3]:
df = pd.read_csv(r"C:\Users\joshu\Downloads\processed\nba_live_wp_final_features.csv")

print(df.shape)
df.head()


(1661962, 29)


,game_id,game_date,home_team_won,period,seconds_remaining,score_diff,is_overtime,minutes_remaining,abs_score_diff,home_possession,score_x_time,abs_score_x_time,score_per_minute_remaining,diff_net_rtg_blend_70_20_10,home_offense_bonus_q4,away_offense_bonus_q4,current_offense_has_bonus_q4,possessions_remaining_est,score_per_possession_remaining,abs_score_per_possession_remaining,score_x_possessions_remaining,abs_score_x_possessions_remaining,home_up_1_to_3_late_q4,home_down_1_to_3_late_q4,home_up_4_to_6_late_q4,home_down_4_to_6_late_q4,home_up_7_to_9_late_q4,home_down_7_to_9_late_q4,tied_late_q4
0,22300001,2023-11-03,1,1,2861.0,2,0,47.683333,2,1,95.366667,95.366667,0.041508,-2.018877,0,0,0,99.340278,0.019932,0.019932,198.680556,198.680556,0,0,0,0,0,0,0
1,22300001,2023-11-03,1,1,2843.0,2,0,47.383333,2,0,94.766667,94.766667,0.041768,-2.018877,0,0,0,98.715278,0.020057,0.020057,197.430556,197.430556,0,0,0,0,0,0,0
2,22300001,2023-11-03,1,1,2840.0,2,0,47.333333,2,1,94.666667,94.666667,0.041812,-2.018877,0,0,0,98.611111,0.020078,0.020078,197.222222,197.222222,0,0,0,0,0,0,0
3,22300001,2023-11-03,1,1,2835.0,2,0,47.250000,2,1,94.500000,94.500000,0.041885,-2.018877,0,0,0,98.437500,0.020113,0.020113,196.875000,196.875000,0,0,0,0,0,0,0
4,22300001,2023-11-03,1,1,2832.0,2,0,47.200000,2,1,94.400000,94.400000,0.041929,-2.018877,0,0,0,98.333333,0.020134,0.020134,196.666667,196.666667,0,0,0,0,0,0,0


## 2. Define target, features, and identifier columns

- `home_team_won` is the label.
- `game_id` / `game_date` are identifiers used for grouping and are **not** features.
- Everything else is a numeric in-game feature already engineered (score diff, time remaining,
  possession, net-rating blend, bonus situation, late-game score-margin buckets, etc.).

XGBoost handles `NaN` natively (it learns a default split direction for missing values), so we
don't need to impute `diff_net_rtg_blend_70_20_10` — we just let the model deal with it.


In [4]:
TARGET = "home_team_won"
ID_COLS = ["game_id", "game_date"]

feature_cols = [c for c in df.columns if c not in ID_COLS + [TARGET]]
print(f"{len(feature_cols)} features:")
for c in feature_cols:
    print(" -", c)

X = df[feature_cols]
y = df[TARGET].astype(int)
groups = df["game_id"]

print("\nMissing values per feature:")
print(X.isna().sum()[X.isna().sum() > 0])


26 features:
 - period
 - seconds_remaining
 - score_diff
 - is_overtime
 - minutes_remaining
 - abs_score_diff
 - home_possession
 - score_x_time
 - abs_score_x_time
 - score_per_minute_remaining
 - diff_net_rtg_blend_70_20_10
 - home_offense_bonus_q4
 - away_offense_bonus_q4
 - current_offense_has_bonus_q4
 - possessions_remaining_est
 - score_per_possession_remaining
 - abs_score_per_possession_remaining
 - score_x_possessions_remaining
 - abs_score_x_possessions_remaining
 - home_up_1_to_3_late_q4
 - home_down_1_to_3_late_q4
 - home_up_4_to_6_late_q4
 - home_down_4_to_6_late_q4
 - home_up_7_to_9_late_q4
 - home_down_7_to_9_late_q4
 - tied_late_q4

Missing values per feature:
diff_net_rtg_blend_70_20_10    6658
dtype: int64


## 3. Train / test split — grouped by game

Rows within the same game are highly correlated (they're the same game state evolving over
time), so a plain random row split would leak information from a game into both train and test.
We hold out entire games instead.


In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

print(f"Train rows: {len(X_train):,}  ({groups_train.nunique()} games)")
print(f"Test rows:  {len(X_test):,}  ({groups.iloc[test_idx].nunique()} games)")


Train rows: 1,413,485  (3276 games)
Test rows:  248,477  (579 games)


## 4. Hyperparameter search

`GroupKFold` again ensures each CV fold's validation games never appear in that fold's training
data. We optimize log loss, since we care about well-calibrated *probabilities*, not just
classification accuracy — a live win-probability model is only useful if 70% actually means
"wins about 70% of the time."


In [ ]:
param_dist = {
    "n_estimators": [200, 400, 600, 800],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.02, 0.05, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 10],
    "gamma": [0, 0.1, 0.5, 1],
    "reg_lambda": [0.5, 1, 2, 5],
    "reg_alpha": [0, 0.1, 0.5, 1],
}

base_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=1,  # single-threaded per fit -- RandomizedSearchCV parallelizes across fits instead
)

import time

# --- Quick sanity-check pass first: 3 folds, 5 candidates -----------------
cv_quick = GroupKFold(n_splits=3)

quick_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=5,
    scoring="neg_log_loss",
    cv=cv_quick.split(X_train, y_train, groups_train),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=2,
)

t0 = time.time()
quick_search.fit(X_train, y_train)
elapsed = time.time() - t0
print(f"\nQuick pass (15 fits) took {elapsed:.1f} sec -> ~{elapsed/15:.1f} sec/fit")
print(f"Estimated full search (200 fits): ~{elapsed/15*200/60:.1f} min")

Fitting 3 folds for each of 5 candidates, totalling 15 fits


## 5. Refit final model on the full training set with best params


In [ ]:
best_params = search.best_params_

final_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **best_params,
)

final_model.fit(X_train, y_train)


## 6. Evaluate on held-out games


In [ ]:
test_probs = final_model.predict_proba(X_test)[:, 1]

ll = log_loss(y_test, test_probs)
auc = roc_auc_score(y_test, test_probs)
brier = brier_score_loss(y_test, test_probs)

print(f"Test log loss:    {ll:.4f}")
print(f"Test AUC:         {auc:.4f}")
print(f"Test Brier score: {brier:.4f}")


In [ ]:
# Calibration curve: does a predicted 70% actually correspond to ~70% win rate?
frac_pos, mean_pred = calibration_curve(y_test, test_probs, n_bins=20, strategy="quantile")

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
plt.plot(mean_pred, frac_pos, marker="o", label="XGBoost model")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed home win rate")
plt.title("Calibration curve")
plt.legend()
plt.tight_layout()
plt.show()


## 7. Feature importance


In [ ]:
importances = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 8))
importances.plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("XGBoost feature importance (gain-based)")
plt.tight_layout()
plt.show()

importances


## 8. Final model function

A single function that takes the live game state (as a dict or DataFrame row with the same
feature names used in training) and returns the home team's win probability. This is what you'd
call from a live scoreboard feed to get a real-time probability at any moment in the game.


In [ ]:
FEATURE_ORDER = feature_cols  # exact column order/names the model expects

def predict_win_probability(game_state, model=final_model, feature_order=FEATURE_ORDER):
    """
    Predict home team win probability at any point in a live NBA game.

    Parameters
    ----------
    game_state : dict or pandas.DataFrame
        - dict: a single moment's feature values, e.g.
          {"period": 4, "seconds_remaining": 125.0, "score_diff": -3, ...}
        - DataFrame: one row per moment, columns matching `feature_order`.
          Missing columns are filled with NaN (XGBoost handles missing values natively).
    model : xgb.XGBClassifier
        Trained model (defaults to the model fit in this notebook).
    feature_order : list[str]
        Column order/names the model was trained on.

    Returns
    -------
    float or np.ndarray
        Home team win probability (or array of probabilities if a DataFrame was passed).
    """
    if isinstance(game_state, dict):
        row = pd.DataFrame([game_state])
        row = row.reindex(columns=feature_order)
        prob = model.predict_proba(row)[:, 1][0]
        return float(prob)

    elif isinstance(game_state, pd.DataFrame):
        rows = game_state.reindex(columns=feature_order)
        return model.predict_proba(rows)[:, 1]

    else:
        raise TypeError("game_state must be a dict or a pandas DataFrame")


### Example: single moment


In [ ]:
example_state = X_test.iloc[0].to_dict()
prob = predict_win_probability(example_state)
print(f"Home win probability: {prob:.1%}")


## 9. Sanity check: win probability trajectory for one game

Plot the model's predicted home win probability across every recorded moment of a single game,
compared to the actual final outcome.


In [ ]:
sample_game_id = groups.iloc[test_idx].iloc[0]
game_rows = df[df["game_id"] == sample_game_id].sort_values(["period", "seconds_remaining"], ascending=[True, False])

game_probs = predict_win_probability(game_rows[feature_cols])

plt.figure(figsize=(10, 4))
plt.plot(range(len(game_probs)), game_probs)
plt.axhline(0.5, linestyle="--", color="gray")
plt.ylim(0, 1)
plt.xlabel("Play index (chronological)")
plt.ylabel("Home win probability")
outcome = "Home won" if game_rows[TARGET].iloc[0] == 1 else "Home lost"
plt.title(f"Game {sample_game_id} — live win probability ({outcome})")
plt.tight_layout()
plt.show()


## 10. Save the model

Persist the trained model (and the feature order it expects) so it can be loaded later without
retraining.


In [ ]:
joblib.dump(
    {"model": final_model, "feature_order": FEATURE_ORDER, "best_params": best_params},
    "nba_live_wp_xgboost_model.joblib",
)
print("Saved to nba_live_wp_xgboost_model.joblib")
